# VEST neutral-beam analysis with NUBEAM

What does a 200 kW hydrogen beam actually do to a VEST plasma -- where is it
absorbed, what fraction reaches the plasma at all, and where does the rest go?

This notebook answers that from one validated NUBEAM run. It keeps three kinds
of number strictly apart, because conflating them is the easiest way to
overstate what a simulation establishes:

| | |
| --- | --- |
| **Machine description** | VEST geometry and beamline hardware, from `mdescr` |
| **Modelling input** | beam energy and power, background profiles -- chosen for this run |
| **NUBEAM output** | heating, current drive, deposition, losses -- what the code computed |

Nothing here is measured beam data. The beam configuration is what this NUBEAM
case assumes, and it is labelled that way throughout.

Reading order is top to bottom: the case, its inputs, then geometry, profiles,
the power budget, and finally what the run does and does not establish.


## 0. Setup

Point `VAFT_NUBEAM_RUN_DIR` at a completed NUBEAM work directory -- one
produced by `external/nubeam/run-local-vest.sh`, or by `vaft.code.nubeam`
directly. Without it the notebook explains what it would show and stops.


In [ ]:
import os
from pathlib import Path

try:
    _ipython = get_ipython()
except NameError:
    _ipython = None
if _ipython is None:
    os.environ.setdefault("MPLBACKEND", "Agg")
elif "IPKernelApp" in _ipython.config:
    _ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np

from vaft.code import nubeam
from vaft.plot import nubeam as nbplot

plt.rcParams["figure.dpi"] = 110

_configured = os.environ.get("VAFT_NUBEAM_RUN_DIR")
RUN_DIR = Path(_configured).expanduser() if _configured else None
HAVE_RUN = RUN_DIR is not None and RUN_DIR.is_dir()

if HAVE_RUN:
    print(f"NUBEAM run: {RUN_DIR}")
else:
    print(
        "Set VAFT_NUBEAM_RUN_DIR to a completed NUBEAM work directory.\n"
        "Every section below reports what it would show and skips."
    )


## 1. The case

`collect_nubeam_outputs` reads a finished run directory without re-running
anything. Every product is optional, so this also tells us which parts of the
analysis this particular run can support.


In [ ]:
result = nubeam.collect_nubeam_outputs(RUN_DIR) if HAVE_RUN else None
native = result.outputs_native if result is not None else None

if native is not None:
    print(f"run id                {native.runid}")
    print(f"radial profiles       {len(native.profiles)}")
    print(f"scalar diagnostics    {len(native.scalars)}")
    print(f"deposition markers    {native.birth.count if native.birth else 'not written'}")
    print(f"lost fast ions        {native.lost.count if native.lost else 'not collected'}")
    print(f"power balance blocks  {len(native.power_balance)}")
    print(f"interpolation warnings {native.interpolation_warnings}")
else:
    print("Would report the run id and which products the run produced.")


## 2. Inputs, by provenance

The beam configuration below is a **modelling input**, not a measurement. Beam
power and energy enter through the `profiles` file that `makeprofile` writes,
and the Plasma State records what NUBEAM actually used -- which is the value to
trust, since `mdescr` separately tabulates energy-fraction data at a different
energy that is easy to misread as the injection energy.


In [ ]:
import warnings

beam = {}
if HAVE_RUN:
    state = RUN_DIR / f"{native.runid}.cdf"
    if state.is_file():
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            import xarray as xr

            with xr.open_dataset(state, decode_times=False) as ds:
                for key in ("kvolt_nbi", "power_nbi"):
                    if key in ds.variables:
                        beam[key] = (
                            float(np.asarray(ds[key].values).ravel()[0]),
                            ds[key].attrs.get("units", ""),
                        )

if beam:
    print("NUBEAM modelling input (from the profiles file, via the Plasma State)")
    for key, (value, unit) in beam.items():
        print(f"   {key:<12s} {value:>10.4g} {unit}")
    print("\nThese are assumptions of this case, not measured beam parameters.")
else:
    print("Would report beam energy and power, labelled as modelling inputs.")


The beamline geometry is a **machine description**. It is still NUBEAM-derived
rather than as-built: reconciling it against the real hardware is issue #265.

One value deserves care. `srtcen` is a *signed* tangency radius -- the sign
carries the injection direction -- so it is reported as written rather than as
a magnitude.


In [ ]:
geometry = {}
if HAVE_RUN:
    import f90nml

    for descriptor in sorted(RUN_DIR.glob("mdescr_*.dat")):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            parsed = f90nml.read(descriptor)
        for body in parsed.values():
            geometry.update(body)

interesting = (
    ("nbi_src_name", "beam source"),
    ("srtcen", "tangency radius [m], signed"),
    ("zbsc", "source elevation [m]"),
    ("nbshape", "source grid shape"),
    ("b_halfwidth", "source half-width [m]"),
    ("b_halfheight", "source half-height [m]"),
    ("b_hdivergence", "horizontal divergence [deg]"),
    ("b_vdivergence", "vertical divergence [deg]"),
)
if geometry:
    for key, description in interesting:
        if key in geometry:
            value = geometry[key]
            value = value[0] if isinstance(value, list) else value
            print(f"   {description:<30s} {value}")
else:
    print("Would report the beamline geometry from mdescr.")


## 3. Where the beam deposits

NUBEAM writes one marker per deposition track. The top view is the natural one
for a tangential beam -- it shows the chord through the plasma, which a
poloidal projection collapses -- so both are shown.

The birth file stores centimetres and degrees, and says so nowhere; the
plotting layer converts and the axes are metres.


In [ ]:
if HAVE_RUN and native.birth is not None:
    figure, axes = plt.subplots(1, 2, figsize=(11, 5))
    nbplot.nubeam_deposition_topview(result, ax=axes[0])
    nbplot.nubeam_deposition_poloidal(result, ax=axes[1])
    figure.tight_layout()
else:
    print("Would show beam deposition from above and in the poloidal plane.")


## 4. Radial profiles

NUBEAM writes these as **per-zone integrals**, not densities: `pbe` is the
watts deposited in a zone. They are plotted in those units rather than divided
by a zone volume, which would make them derived quantities.

The abscissa is NUBEAM's own `rho`, which is toroidal-flux based.


In [ ]:
rho = None
if HAVE_RUN:
    state = RUN_DIR / f"{native.runid}.cdf"
    if state.is_file():
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            with xr.open_dataset(state, decode_times=False) as ds:
                if "rho_nbi" in ds.variables:
                    rho = np.asarray(ds["rho_nbi"].values)

wanted = ["pbe", "pbi", "curbeam", "nbeami"]
available = [q for q in wanted if HAVE_RUN and q in native.profiles]
missing = [q for q in wanted if q not in available]

if available:
    figure, axes = plt.subplots(2, 2, figsize=(11, 7))
    for panel, quantity in zip(axes.ravel(), available):
        nbplot.nubeam_profile(result, quantity, rho=rho, ax=panel)
    for panel in axes.ravel()[len(available):]:
        panel.set_visible(False)
    figure.tight_layout()
if missing:
    print(f"absent from this run: {', '.join(missing)}")
if not available:
    print("Would show beam heating, driven current and fast ion density.")


A hydrogen plasma makes no fusion products, so `pfuse`, `pfusi` and `curfusn`
are legitimately absent rather than zero. NUBEAM says so itself in the step
log. Asking for one reports that rather than drawing a flat line at zero.


In [ ]:
if HAVE_RUN:
    try:
        nbplot.build_nubeam_profile(result, "pfuse")
    except nbplot.NUBEAMPlotError as error:
        print(error)
else:
    print("Would demonstrate that an absent quantity is reported, not zero-filled.")


## 5. The power budget

NUBEAM closes its own energy accounting at the end of each step, including the
residual. This is read from the step log rather than reconstructed from the
profiles -- a reconstruction would be a different number carrying different
assumptions.

This is the single most informative output for a small tokamak, because it
says directly what fraction of the injected power the plasma actually absorbs.


In [ ]:
if HAVE_RUN and native.power_balance:
    figure, _ = nbplot.nubeam_power_accounting(result)
    figure.tight_layout()

    balance = native.power_balance[0]
    fractions = balance.fractions()
    heating = sum(v for k, v in fractions.items() if "heating" in k)
    print(f"coupled to the plasma as heating: {100 * heating:.1f}%")
    for channel in ("shine-through", "bad orbit loss"):
        if channel in fractions:
            print(f"lost to {channel:<18s}      {100 * fractions[channel]:.1f}%")
else:
    print("Would show NUBEAM's power balance and the absorbed fraction.")


## 6. Lost fast ions

Where NUBEAM stopped following markers. `lstype` separates prompt loss -- ions
lost before completing an orbit -- from orbit loss, and the split matters:
NUBEAM's log labels the whole channel "bad orbit loss" regardless.

These coordinates are already metres, unlike the deposition markers.


In [ ]:
if HAVE_RUN and native.lost is not None and native.lost.count:
    print(f"channels: {native.lost.channel_counts()}")
    figure, _ = nbplot.nubeam_lost_fast_ions(result)
    figure.tight_layout()
elif HAVE_RUN:
    print("This run collected no lost-particle record.")
else:
    print("Would show where fast ions were lost, split by loss channel.")


## 7. What this run establishes

Read the numbers above against these limits.

**It does establish** how the modelled beam distributes its power for this
equilibrium and these background profiles: the absorbed fraction, the split
between electron and ion heating, the driven current, and where deposition and
losses occur spatially.

**It does not establish** anything about the real beam. The energy, power and
energy-fraction structure are inputs to this case. Until the as-built beam data
are reconciled (issue #265), a change in these results tracks a change in the
assumptions as readily as a change in the physics.

**Two numbers deserve scepticism.** Monte Carlo noise is large where zone
volumes are small, so near-axis profile structure needs a particle-count scan
before it is believed. And the FRANTIC halo and recombination channels sit
15-28% away from the shipped reference cases against a much smaller noise
floor, so neutral-related quantities are the least trustworthy outputs here --
see `external/nubeam/VALIDATION.md`.
